In [13]:
import pandas as pd
from music21 import key
import mir_eval
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [14]:
ROOTS = ['C', 'Db', 'D', 'Eb', 'E', 'F', 'F#', 'G', 'Ab', 'A', 'Bb', 'B']

def normalize_key(key_str):
    """
    Convert a key string to a canonical form.
    E.g., "Gb major" -> "F# major"
    Enharmonically equivalent keys map to the same representation.
    """
    try:
        key_str = key_str.replace(":", " ")
        raw_tonic, raw_mode = key_str.split()
        k = key.Key(raw_tonic, raw_mode.lower())
        # Get the pitch and convert to sharp representation (canonical form)
        tonic = k.tonic
        pc = tonic.pitchClass  # 0-11 representing pitch class

        canonical_tonic = ROOTS[pc]
        mode = k.mode.lower()
        
        return f"{canonical_tonic} {mode}"
    except Exception as e:
        print(e)

In [15]:
def get_avg_weighted_score(y_true, y_pred):
    """
    Calculates the MIREX weighted score for each prediction and returns the average score across all predictions.

    See mirex website for more details: https://music-ir.org/mirex/wiki/2025:Audio_Key_Detection
    """
    cumulative_score = 0.0

    for true, pred in zip(y_true, y_pred):
        weighted_score = mir_eval.key.weighted_score(true, pred)
        cumulative_score += weighted_score

    avg_score = cumulative_score / len(y_true)
    return avg_score

def get_metrics(predicted_keys, true_keys):
    """
    Computes various evaluation metrics for key estimation.
    """
    return {
        'Accuracy': accuracy_score(true_keys, predicted_keys),
        'Precision (Macro)': precision_score(true_keys, predicted_keys, average='macro'),
        'Recall (Macro)': recall_score(true_keys, predicted_keys, average='macro'),
        'F1 Score (Macro)': f1_score(true_keys, predicted_keys, average='macro'),
        'Average MIREX Score': get_avg_weighted_score(true_keys, predicted_keys),
    }

def print_metrics(metrics):
    for metric_name, value in metrics.items():
        print(f"{metric_name}: {value:.4f}")

In [16]:
poc_df = pd.read_csv("poc.csv")
track_ids = poc_df["id"].tolist()
actual_keys_dict = dict(zip(poc_df["id"], poc_df["key"]))
actual_keys = [normalize_key(k) for k in actual_keys_dict.values()]

In [17]:
preds_df = pd.read_csv("llm-predictions.csv")
pred_keys_dict = dict(zip(preds_df["track_id"], preds_df["key"]))
pred_keys = [normalize_key(k) for k in pred_keys_dict.values()]

In [18]:
metrics = get_metrics(pred_keys, actual_keys)
print_metrics(metrics)

Accuracy: 0.7700
Precision (Macro): 0.4873
Recall (Macro): 0.7153
F1 Score (Macro): 0.4970
Average MIREX Score: 0.8213


/Users/jeffreygordon/Dev/llm-key-estimation/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/jeffreygordon/Dev/llm-key-estimation/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [22]:
chords_dict = dict(zip(poc_df["id"], poc_df["chords"]))
explanations_dict = dict(zip(preds_df["track_id"], preds_df["explanation"]))
wrong_ids = [tid for tid in track_ids if normalize_key(actual_keys_dict[tid]) != normalize_key(pred_keys_dict[tid])]

wrong_df = pd.DataFrame([
    {
        "track_id": tid,
        "actual_key": normalize_key(actual_keys_dict[tid]),
        "predicted_key": normalize_key(pred_keys_dict[tid]),
        "explanation": explanations_dict[tid],
        "chords": chords_dict[tid],
    }
    for tid in wrong_ids
])

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)
display(wrong_df)

,track_id,actual_key,predicted_key,explanation,chords
0,2014,C major,F major,"The sequence contains frequent returns to F major, D minor, and C major, which all fit within the F major scale. While there is some chromaticism, these diatonic chords anchor the progression in F major.","F#:hdim7 2.0s, C:maj/5 1.0s, Eb:maj 1.0s, F#:hdim7 2.0s, C:maj/5 1.0s, A:maj 1.0s, D:min 1.0s, A:min/4 0.5s, D:min 0.5s, A:min7/4 0.5s, D:min 0.5s, A:min/4 1.0s, F#:hdim7 1.0s, G:(5) 1.0s, F:maj/9 1.0s, G:maj 1.0s, F:maj 4.0s, F:maj7 1.5s, F:maj6 0.5s, F#:hdim7 2.0s, F:maj 4.0s, F:maj/9 2.0s, C:maj 2.0s"
1,2084,Eb major,C major,"The harmonic sequence centers on C major, with repeated cadences resolving to C major throughout the piece. The progression frequently utilizes modal interchange chords such as C minor 6th and F minor 6th, which are characteristic of functional harmony in the key of C major.","Ab:min7 2.0s, G:7 2.0s, Gb:7 2.0s, F:hdim7 2.0s, Fb:maj7 4.0s, D:min7 2.0s, Bb:7 2.0s, C:maj6 2.0s, C:min6 2.0s, G:min6 2.0s, A:7 1.5s, A:7 0.5s, F:min6 2.0s, G:7 1.5s, G:7 0.5s, C:maj 2.0s, D:(b3,5,b7,9,11) 1.5s, G:(3,5,b7,9,b9,11,13) 0.5s, C:maj6 2.0s, C:min6 2.0s, G:min6 2.0s, A:7 1.5s, A:7 0.5s, F:min6 2.0s, G:7 1.5s, G:7 0.5s, C:maj 1.0s, Ab:dim7 1.0s, C:maj6 1.0s, C:aug 1.0s, F:maj6 1.0s, F:7 1.0s, F:min6 1.5s, G:7(b9) 0.5s, C:maj 2.0s, G:min7 2.0s, F:maj6 1.0s, F:7 1.0s, F:min6 1.5s, G:7(b9) 0.5s"
2,2143,Bb major,C minor,"The chord sequence consistently revolves around C minor, frequently using chords like C:min9, C:min7, F:9, and G:7. Although there are brief modulations to other keys like Eb Major and D Major, the frequent return to C minor establishes it as the primary key of the piece.","C:min9 1.0s, F:9 0.5s, C:min7 0.5s, D:min7 1.0s, Db:dim 1.0s, C:min7 1.0s, F:aug 1.0s, D:hdim7 1.0s, G:7 1.0s, B:dim 1.0s, C:hdim7 1.0s, D:min7 1.0s, G:7 1.0s, C:min7 0.8s, B:7 1.0s, Bb:maj 2.2s, C:min9 1.0s, F:9 0.5s, C:min7 0.5s, D:min7 1.0s, Db:dim 1.0s, C:min7 1.0s, F:aug 1.0s, D:hdim7 1.0s, G:7 1.0s, B:dim 1.0s, C:hdim7 1.0s, D:min7 1.0s, G:7 1.0s, C:min7 0.8s, B:7 1.0s, Bb:maj6 2.2s, F:min7 1.0s, Bb:(3,#5,b7) 1.0s, Eb:maj7 1.5s, Eb:maj6 0.5s, F:min7 1.0s, Bb:7 1.0s, Eb:maj9 1.5s, Eb:maj6 0.5s, E:min7 1.0s, A:7(#5,b9) 1.0s, D:maj9 1.5s, D:maj6 0.5s, G:min 0.5s, D:aug 0.5s, G:min7 0.5s, C:7 0.5s, C:min7 1.0s, F:7 1.0s, C:min9 1.0s, F:9 0.5s, C:min7 0.5s, D:min7 1.0s, Db:dim 1.0s, C:min7 1.0s, F:(3,#5,b7) 1.0s, D:hdim7 1.0s, G:7 1.0s, B:dim 1.0s, C:hdim7 1.0s, D:min7 1.0s, G:9 1.0s"
3,1668,Ab major,F minor,"The progression consistently centers on F minor, utilizing the C7 chord as a dominant to resolve back to the tonic. This harmonic pattern, combined with the recurring F minor chords throughout the sequence, clearly establishes F minor as the key.","F:min 3.0s, C:7 1.0s, F:min 4.0s, Bb:min7 4.0s, F:min 4.0s, Ab:min7 1.0s, Db:9 2.0s, C:7 1.0s, F:min 4.0s, F:min 3.0s, C:7 1.0s, F:min 4.0s, Bb:min7 4.0s, F:min 4.0s, Ab:min7 1.0s, Db:9 2.0s, C:7 1.0s, F:min 4.0s"
4,5859,C major,G major,"The chord sequence is dominated by G major and D7 chords, which act as the I and V chords in the key of G major. The occasional inclusion of the C major chord, which functions as the IV chord, further reinforces G major as the key.","G:maj 2.0s, D:7 4.0s, G:maj 4.0s, D:7 4.0s, G:maj 4.0s, D:7 2.0s, G:maj 2.0s, C:maj 2.0s, G:maj 1.0s, D:7 1.0s, G:maj 1.0s, D:7 3.0s, G:maj 4.0s, D:7 4.0s, G:maj 4.0s, D:7 4.0s, G:maj 4.0s, D:7 2.0s, G:maj 2.0s, C:maj 2.0s, G:maj 1.0s, D:7 1.0s, G:maj 1.0s, D:7 1.0s"
...,...,...,...,...,...
225,5350,C major,A minor,"The chord progression centers repeatedly on A minor, supported by the frequent use of the tonic (A minor) and the dominant (E major) chords. The inclusion of chords such as F major, G major, C major, and D minor is consistent with the A natural and harmonic minor scales, further establishing A minor as the key.","A:min 2.0s, E:min 2.0s, D:min 2.0s, A:min 2.0s, E:min 2.0s, A:min 2.0s, F:maj 1.0s, G:maj 1.0s, A:min 1.0s, G:9 1.0s, C:m